# Keeping Incorrect

In [2]:
# these warnings are fine. you can ignore them.
import random, math

import sys
sys.path.append('../')
from util import *

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Imports finished.")

Imports finished.


## Setting Up Dataset/Model/Ground Truth

In [4]:
dataset = Dataset(root='/tmp/Cora', name='Cora', device=device)
data, in_feats, h_feats, num_classes = dataset.get_data()

model = get_model('../', in_feats, h_feats, num_classes, "cora", Models.GCN)
ground_truth, orig_pred = get_ground_truth(model, data, Models.GCN, testMask=True)
print(ground_truth)

0.745


In [5]:
adj = data.x

## Experiments

**Note:** The ground truth here is $0.745$

In [6]:
class_set = {label: [] for label in set(data.y.tolist())}

# test_indices = torch.nonzero(data.test_mask, as_tuple=False).squeeze()
G, x, y, train_mask, test_mask = convert_to_networkx(dataset.get_data()[0])
for i in G.nodes():
    class_set[data.y[i].item()].append(i)

# print("This is the dictionary containing each class and its respective vertices:\n\t", homophilic_set)

In [7]:
G, x, y, train_mask, test_mask = convert_to_networkx(data)

edge_homo = [0] * len(G.nodes())

for n in G.nodes():
    edge_homo[n] = get_node_homophily(G, n, y, y[n].item())

In [8]:
def get_lowest_degree_nodes(graph):
    degrees = dict(graph.degree())
    sorted_degrees = dict(sorted(degrees.items(), key=lambda item: item[1]))    
    return sorted_degrees

In [9]:
def get_highest_homophily_in_each_class(G, s, e):
    res = {key: (-1, float('-inf')) for key in range(7)}
    for i in range(0, 7):
        for j in s[i]:
            val = (1 + e[j])
            # val = (1 + e[j]) * G.degree(j)
            if val > res[i][1]:
                res[i] = tuple([j, val])

    return res

In [10]:
def get_max_tuple_excluding_key(i, data):
    filtered_data = {key: value for key, value in data.items() if key != i}
    
    max_tuple = max(filtered_data.values(), key=lambda x: x[1])
    
    return max_tuple

In [11]:
def get_lowest_homophily_in_each_class(G, s, e):
    res = {key: (-1, float('inf')) for key in range(7)}
    for i in range(0, 7):
        for j in s[i]:
            val = (1 + e[j]) * G.degree(j)
            if val < res[i][1]:
                res[i] = tuple([j, val])

    return res

In [12]:
def check_not_in_neighborhood(G, i, j, y):
    neighbors = G.neighbors(j)
    for n in neighbors:
        if n != i and y[n].item() == y[i].item():
            return False
    return True

In [13]:
def check_incorrect_neighborhood(G, i, j, pred, y):
    neighbors = G.neighbors(j)
    for n in neighbors:
        if y[n].item() == pred[n].item():
            return False
    return True

In [18]:
data = dataset.get_data()[0]
    
G, x, y, train_mask, test_mask = convert_to_networkx(data)

# get incorrect classification indices
incorrect = torch.nonzero(orig_pred != y).squeeze().tolist()

# sort based on minimum degree
lowest_homophily_nodes = {}
for j in range(0, len(incorrect)):
    if (edge_homo[j] <= 1):
        lowest_homophily_nodes[j] = edge_homo[j]

# lowest_homophily_nodes = dict(sorted(lowest_homophily_nodes.items(), key=lambda x: x[1], reverse=False))
# zero_homophily_nodes = {key: value for key, value in my_dict.items() if value == 0}
print(len(lowest_homophily_nodes))

# put all of these in a set for a budget and select
tuple_list = []
for a in range(0, len(lowest_homophily_nodes)):
    i = list(lowest_homophily_nodes.keys())[a]
    for b in range(a + 1, len(lowest_homophily_nodes)):
        j = list(lowest_homophily_nodes.keys())[b]
        # if i != j and not G.has_edge(i, j) and orig_pred[j] != y[i] and orig_pred[j] != orig_pred[i]:
        if i != j and not G.has_edge(i, j) and not G.has_edge(j, i) and orig_pred[j] != y[i] and check_not_in_neighborhood(G, i, j, y):
            tuple_list.append((i, j))

# print(tuple_list)
tuple_list = sorted(tuple_list, key=lambda x: edge_homo[x[0]] + edge_homo[x[1]])
print(len(tuple_list))

665
163784


In [19]:
max_b = len(tuple_list) / (G.number_of_edges() * 1/2)

In [20]:
ptb_rate = [(i+1) * 0.05 for i in range(int(max_b / 0.05) + 1)]
print(ptb_rate)


# print_graph(G)
results = []
for ptb in ptb_rate:
    data = dataset.get_data()[0]
    modified_graph = data
    
    init_edges = len(modified_graph.edge_index[1])
    
    G, x, y, train_mask, test_mask = convert_to_networkx(modified_graph)
    
    # budget = len(tuple_list)
    budget = math.floor(ptb * G.number_of_edges() * 1/2)
    # print(budget)
    # print(len(tuple_list))
    
    # random.shuffle(tuple_list)
    
    edges_to_add = tuple_list[:budget]
    
    for i in edges_to_add:
        one = i[0]
        two = i[1]
    
        if G.has_edge(one, two):
            print('uh oh')
        add_edge(G, one, two, undirected=True)
    
    modified_graph = convert_to_pyg(G, x, y, train_mask, test_mask)
    final_edges = len(modified_graph.edge_index[1])
    
    acc, pred = test_model(model, modified_graph, GCNtype=Models.GCN, testMask=True)
    output_accuracy_change(ground_truth, acc) 
    number_added_edges(init_edges, final_edges, is_undirected=True)
    results.append(abs(acc - ground_truth))

# print_graph(G)
# print("Number of k-hops considered:", z)

[0.05, 0.1, 0.15000000000000002, 0.2, 0.25, 0.30000000000000004, 0.35000000000000003, 0.4, 0.45, 0.5, 0.55, 0.6000000000000001, 0.65, 0.7000000000000001, 0.75, 0.8, 0.8500000000000001, 0.9, 0.9500000000000001, 1.0, 1.05, 1.1, 1.1500000000000001, 1.2000000000000002, 1.25, 1.3, 1.35, 1.4000000000000001, 1.4500000000000002, 1.5, 1.55, 1.6, 1.6500000000000001, 1.7000000000000002, 1.75, 1.8, 1.85, 1.9000000000000001, 1.9500000000000002, 2.0, 2.0500000000000003, 2.1, 2.15, 2.2, 2.25, 2.3000000000000003, 2.35, 2.4000000000000004, 2.45, 2.5, 2.5500000000000003, 2.6, 2.6500000000000004, 2.7, 2.75, 2.8000000000000003, 2.85, 2.9000000000000004, 2.95, 3.0, 3.0500000000000003, 3.1, 3.1500000000000004, 3.2, 3.25, 3.3000000000000003, 3.35, 3.4000000000000004, 3.45, 3.5, 3.5500000000000003, 3.6, 3.6500000000000004, 3.7, 3.75, 3.8000000000000003, 3.85, 3.9000000000000004, 3.95, 4.0, 4.05, 4.1000000000000005, 4.15, 4.2, 4.25, 4.3, 4.3500000000000005, 4.4, 4.45, 4.5, 4.55, 4.6000000000000005, 4.65, 4.7, 

In [ ]:
# Plotting
plt.plot(ptb_rate, results, label="accuracy change")

plt.xlabel('PTB Rates')
plt.ylabel('Change in Accuracy')
plt.title('Attacks on CORA Dataset')
plt.legend()

output_path = f"./data/keeping_incorrect_cora"
plt.savefig(output_path)

# Show plot
plt.show()

In [ ]:
import os

def save_data(file_path, data):
    # Automatically create the directory if it doesn't exist
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    # Save the data to the file
    np.save(file_path, data)

# Assuming 'dataset' is a variable you have already defined
dataset_paths = [
    f'./data-arrays/cora/keeping_incorrect_cora.npy',
]

results = [
    np.array(gcn_vals),
    np.array(gat_vals),
    np.array(jaccard_vals),
    np.array(gsage_vals),
    np.array(gsaint_vals),
    np.array(ptb_rates)
]

# Save each results array to its corresponding file
for path, data in zip(dataset_paths, results):
    save_data(path, data)